# Hourly HSS catalogs — SDDS B1

Create an HSS list for each satellite parquet selected below. Inputs must contain **hourly scalar solar-wind speed (km/s), proton density (cm⁻³), and proton temperature (K)** on a datetime index. Column names are mapped explicitly; naive timestamps are interpreted as UTC. Missing timestamps are inserted; finer-cadence data are rejected rather than averaged.

**Method.** Flag hours with `speed > 400`, `density < 6.1`, and `temperature > 82000`. Preserve these flags and additionally flag any hour with **more than 12 initially positive samples in its centered 25-sample window** (±12 h). Group consecutive flags and retain events spanning ≥24 h, or ≥12 h with peak speed **>450 km/s**. Report the first/last retained timestamps and the earliest time of maximum speed.

**Provenance.** Python port of `sdds/downloader/dbscripts/hss/modelB1.sql` and its dependency `ace_fill.sql`, read at SDDS Git revision `4f32f0ae0c3094fedfe7945a31c1fcadc66b3020`. The files last changed in commits `80e2b96f744bd66c9010ab5f045d27cec7fb567b` (B1) and `8c6f4225f19e5ac61acf034317ea3260edd0057c` (fill). Local sources: [B1](/Users/aosh/Developer/sdds/downloader/dbscripts/hss/modelB1.sql), [gap fill](/Users/aosh/Developer/sdds/downloader/dbscripts/hss/ace_fill.sql). The deployed frontend at `smdc@andra.sinp.msu.ru:~/swxdev/models/hss` identifies its embedded ACE catalog as `csv/modelB1hss.ts` (inspected 2026-10-28). This is a repository heuristic; no literature attribution is asserted here.

The ACE thresholds are applied unchanged to every satellite: **no radial scaling, Earth propagation, ICME exclusion, or coronal-hole association**. Different input products need not reproduce the historical deployed ACE catalog.

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

## Inputs and outputs

By default, process all five merged archive parquets. For one satellite, replace `parquet_paths` with `[Path("/absolute/path/to/satellite.parquet")]`. Change `source_columns` for another schema. Inputs must already have invalid/fill values cleaned to NaN; nonpositive or infinite plasma values fail validation. All-null plasma columns remain missing and cannot supply measured HSS evidence.

Each input produces a CSV plus a JSON provenance record under `Outputs/HSS B1`; rerunning replaces those generated files. Source parquets are read only.

In [2]:
archive_dir = Path("/Users/aosh/Developer/Shock-and-Awe/Data/CDAWeb Archive/merged")
parquet_paths = sorted(archive_dir.glob("*.parquet"))
source_columns = {"speed": "V", "density": "N_p", "temperature": "T_p"}

project_root = Path.cwd().resolve()
if project_root.name == "Analysis":
    project_root = project_root.parent
assert (project_root / "Library").is_dir(), "Run from the helio-n root or Analysis directory."
output_dir = project_root / "Outputs" / "HSS B1"

assert parquet_paths, "No input parquets selected."
assert len({path.stem for path in parquet_paths}) == len(parquet_paths), "Input stems must be unique."
parquet_paths

[PosixPath('/Users/aosh/Developer/Shock-and-Awe/Data/CDAWeb Archive/merged/ace_20100101T000000_20260701T000000.parquet'),
 PosixPath('/Users/aosh/Developer/Shock-and-Awe/Data/CDAWeb Archive/merged/psp_20100101T000000_20260701T000000.parquet'),
 PosixPath('/Users/aosh/Developer/Shock-and-Awe/Data/CDAWeb Archive/merged/solo_20100101T000000_20260701T000000.parquet'),
 PosixPath('/Users/aosh/Developer/Shock-and-Awe/Data/CDAWeb Archive/merged/stereo-a_20100101T000000_20260701T000000.parquet'),
 PosixPath('/Users/aosh/Developer/Shock-and-Awe/Data/CDAWeb Archive/merged/stereo-b_20100101T000000_20260701T000000.parquet')]

## B1 implementation

The legacy fill treats each plasma column separately. Its group counter increments only when the current and previous samples are both present. Missing samples in groups of **2–25 rows with valid first/last endpoints** are linearly interpolated; other missing values become **zero**. For an ordinary isolated bounded gap, this allows up to 23 missing hours. This unusual grouping is preserved, including groups with several gaps separated by isolated valid samples. Zero-filled density can pass the low-density test; `missing_plasma_hours` therefore records retained event hours lacking any original plasma input, including inserted timestamps.

SQL window operations without an explicit order are interpreted chronologically here; numerical interpolation uses hour positions instead of large epoch timestamps. This preserves the intended algorithm, without promising bit-for-bit database floating-point results. As in SQL's `s/b/e` labels, isolated positive hours and the first/last input-grid samples are excluded before measuring events. Boundary-crossing events may consequently be truncated; process the full available interval before selecting events.

In [3]:
def find_hss_b1(plasma):
    """Apply the chronological SDDS B1 algorithm to hourly plasma columns."""
    assert isinstance(plasma.index, pd.DatetimeIndex), "A datetime index is required."
    assert not plasma.empty, "Input plasma is empty."
    assert not plasma.index.hasnans, "Timestamps contain NaT."
    assert plasma.index.is_unique, "Duplicate timestamps must be resolved upstream."
    plasma = plasma.loc[:, ["speed", "density", "temperature"]].copy()
    plasma.index = pd.to_datetime(plasma.index, utc=True)
    plasma = plasma.sort_index().astype(float)
    assert (plasma.index == plasma.index.floor("h")).all(), "Use timestamps aligned to whole UTC hours."
    assert not ((plasma <= 0) | np.isinf(plasma)).any().any(), "Clean invalid plasma values to NaN first."
    plasma = plasma.reindex(pd.date_range(plasma.index.min(), plasma.index.max(), freq="h"))

    # Port ace_fill.sql's group definition and endpoint tests, including zero fill.
    filled = plasma.copy()
    positions = pd.Series(np.arange(len(plasma)), index=plasma.index)
    for column in plasma.columns:
        values = plasma[column].to_numpy()
        present = plasma[column].notna()
        groups = (present & present.shift(1, fill_value=False)).cumsum()
        grouped_positions = positions.groupby(groups)
        first = grouped_positions.transform("min").to_numpy()
        last = grouped_positions.transform("max").to_numpy()
        count = grouped_positions.transform("size").to_numpy()
        missing = np.isnan(values)
        interpolate = missing & (count > 1) & (count < 26) & np.isfinite(values[first]) & np.isfinite(values[last])
        result = values.copy()
        result[missing] = 0.0
        result[interpolate] = values[first[interpolate]] + (
            values[last[interpolate]] - values[first[interpolate]]
        ) * (positions.to_numpy()[interpolate] - first[interpolate]) / (last[interpolate] - first[interpolate])
        filled[column] = result

    raw = (filled.speed > 400) & (filled.density < 6.1) & (filled.temperature > 82000)
    marked = raw | (raw.rolling(25, center=True, min_periods=1).sum() > 12)
    groups = (~marked).cumsum()

    # Reproduce SQL's s/b/e labels: adjacent positives, with both neighbors defined.
    retained = marked & (marked.shift(1, fill_value=False) | marked.shift(-1, fill_value=False))
    retained.iloc[0] = False
    retained.iloc[-1] = False
    selected = filled.loc[retained, ["speed"]].copy()
    selected["dt"] = selected.index
    selected["group"] = groups.loc[retained]
    selected["missing_plasma"] = plasma.isna().any(axis=1).loc[retained]
    events = selected.groupby("group", sort=False).agg(
        start_dt=("dt", "min"),
        end_dt=("dt", "max"),
        vmax_dt=("speed", "idxmax"),
        vmax=("speed", "max"),
        samples=("speed", "size"),
        missing_plasma_hours=("missing_plasma", "sum"),
    )
    events["duration_h"] = (events.end_dt - events.start_dt).dt.total_seconds() / 3600
    keep = (events.samples >= 12) & (
        (events.duration_h >= 24) | ((events.duration_h >= 12) & (events.vmax > 450))
    )
    events = events.loc[keep].reset_index(drop=True)
    events.attrs["hourly_rows"] = len(plasma)
    events.attrs["missing_input_hours"] = int(plasma.isna().any(axis=1).sum())
    return events

## Generate catalogs

Times are UTC. `duration_h` is `end_dt − start_dt`, not the sample count: 25 consecutive hourly samples span 24 hours. `vmax` is in km/s. JSON sidecars record the exact input file hash, coverage, column mapping, method, and execution time. The source file stem identifies each catalog; it does not imply a propagation target.

In [4]:
output_dir.mkdir(parents=True, exist_ok=True)
catalogs = {}
summary_rows = []
for parquet_path in parquet_paths:
    source = pd.read_parquet(parquet_path, columns=list(source_columns.values()))
    plasma = source.rename(columns={value: key for key, value in source_columns.items()})
    events = find_hss_b1(plasma)
    catalog_path = output_dir / f"{parquet_path.stem}_hss_b1.csv"
    provenance = {
        "method": "SDDS B1",
        "sdds_revision": "4f32f0ae0c3094fedfe7945a31c1fcadc66b3020",
        "method_sources": ["downloader/dbscripts/hss/modelB1.sql", "downloader/dbscripts/hss/ace_fill.sql"],
        "thresholds": {"speed_gt_km_s": 400, "density_lt_cm3": 6.1, "temperature_gt_K": 82000},
        "bridging": "Keep raw positives; add hours with >12 raw positives in centered 25-hour window.",
        "acceptance": "samples >=12 and (duration_h >=24 or (duration_h >=12 and vmax >450 km/s))",
        "gap_policy": "SQL groups: interpolate missing values with valid endpoints and 2–25 rows; otherwise zero-fill.",
        "sql_interpretation": "Chronological windows; interpolation in hour positions; s/b/e boundary exclusion retained.",
        "cross_satellite_policy": "Fixed ACE thresholds; no radial scaling or propagation.",
        "input_parquet": str(parquet_path.resolve()),
        "input_sha256": hashlib.sha256(parquet_path.read_bytes()).hexdigest(),
        "input_attrs": source.attrs,
        "source_columns": source_columns,
        "units": {"speed": "km/s", "density": "cm^-3", "temperature": "K", "time": "UTC"},
        "coverage_start_utc": str(pd.to_datetime(source.index, utc=True).min()),
        "coverage_end_utc": str(pd.to_datetime(source.index, utc=True).max()),
        "input_rows": len(source),
        **events.attrs,
        "events": len(events),
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "output_csv": str(catalog_path),
    }
    events.to_csv(catalog_path, index=False)
    catalog_path.with_suffix(".json").write_text(json.dumps(provenance, indent=2) + "\n")
    catalogs[parquet_path.stem] = events
    summary_rows.append({
        "source": parquet_path.stem,
        "start_utc": provenance["coverage_start_utc"],
        "end_utc": provenance["coverage_end_utc"],
        "hourly_rows": provenance["hourly_rows"],
        "missing_input_hours": provenance["missing_input_hours"],
        "events": len(events),
        "events_with_missing_input": int((events.missing_plasma_hours > 0).sum()),
    })

summary = pd.DataFrame(summary_rows).set_index("source")
summary

,start_utc,end_utc,hourly_rows,missing_input_hours,events,events_with_missing_input
source,,,,,,
ace_20100101T000000_20260701T000000,2010-01-01 00:00:00+00:00,2026-06-30 23:00:00+00:00,144600,36748,672,387
psp_20100101T000000_20260701T000000,2018-08-12 09:00:00+00:00,2026-06-30 23:00:00+00:00,69111,46190,0,0
solo_20100101T000000_20260701T000000,2020-02-10 06:00:00+00:00,2026-06-30 23:00:00+00:00,55986,18400,105,38
stereo-a_20100101T000000_20260701T000000,2010-01-01 00:00:00+00:00,2026-06-30 23:00:00+00:00,144600,14457,603,138
stereo-b_20100101T000000_20260701T000000,2010-01-01 00:00:00+00:00,2014-09-27 16:00:00+00:00,41537,1440,153,28


## HSS lists

The combined table below is available as `hss_events`; individual tables are in `catalogs` and the exported CSVs. An event's `missing_plasma_hours > 0` identifies reliance on incomplete original input; even a zero count does not establish an ICME-free or coronal-hole origin. Sparse plasma coverage, particularly away from 1 AU, limits interpretation.

In [5]:
hss_events = pd.concat(catalogs, names=["source", "event"])
hss_events

start_dt  \
source                                   event                             
ace_20100101T000000_20260701T000000      0     2010-01-11 16:00:00+00:00   
                                         1     2010-01-13 07:00:00+00:00   
                                         2     2010-01-20 23:00:00+00:00   
                                         3     2010-02-01 10:00:00+00:00   
                                         4     2010-02-18 12:00:00+00:00   
...                                                                  ...   
stereo-b_20100101T000000_20260701T000000 148   2014-08-27 21:00:00+00:00   
                                         149   2014-09-02 16:00:00+00:00   
                                         150   2014-09-07 11:00:00+00:00   
                                         151   2014-09-14 00:00:00+00:00   
                                         152   2014-09-16 13:00:00+00:00   

                                                                  end_dt  \
source                                   event                             
ace_20100101T000000_20260701T000000      0     2010-01-12 04:00:00+00:00   
                                         1     2010-01-14 17:00:00+00:00   
                                         2     2010-01-21 22:00:00+00:00   
                                         3     2010-02-03 17:00:00+00:00   
                                         4     2010-02-19 14:00:00+00:00   
...                                                                  ...   
stereo-b_20100101T000000_20260701T000000 148   2014-08-30 01:00:00+00:00   
                                         149   2014-09-05 09:00:00+00:00   
                                         150   2014-09-09 09:00:00+00:00   
                                         151   2014-09-15 06:00:00+00:00   
                                         152   2014-09-17 17:00:00+00:00   

                                                                 vmax_dt  \
source                                   event                             
ace_20100101T000000_20260701T000000      0     2010-01-11 18:00:00+00:00   
                                         1     2010-01-14 01:00:00+00:00   
                                         2     2010-01-21 00:00:00+00:00   
                                         3     2010-02-03 02:00:00+00:00   
                                         4     2010-02-19 08:00:00+00:00   
...                                                                  ...   
stereo-b_20100101T000000_20260701T000000 148   2014-08-29 05:00:00+00:00   
                                         149   2014-09-03 10:00:00+00:00   
                                         150   2014-09-08 14:00:00+00:00   
                                         151   2014-09-14 02:00:00+00:00   
                                         152   2014-09-17 05:00:00+00:00   

                                                      vmax  samples  \
source                                   event                        
ace_20100101T000000_20260701T000000      0      492.489990       13   
                                         1      513.219971       35   
                                         2      510.459991       24   
                                         3      557.679993       56   
                                         4      487.769989       27   
...                                                    ...      ...   
stereo-b_20100101T000000_20260701T000000 148    446.000000       53   
                                         149    772.599976       66   
                                         150    541.900024       47   
                                         151    526.299988       31   
                                         152    431.200012       29   

                                                missing_plasma_hours  \
source                                   event                         
ace_20100101T000000_20260701T